# 01 · Embedding CLIP + chỉ mục FAISS (Bước 2.3) — Runtime: T4 GPU
Mất phiên thì chạy lại ô 1 → 4 rồi ô 6: shard đã xong được bỏ qua.
Giữ tab mở; đóng tab hoặc để máy ngủ lâu sẽ mất phiên.

In [ ]:
# Ô 1 — Drive và GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Ô 2 — code (lần đầu clone, các lần sau pull) và thư viện
import os
REPO_URL = "https://github.com/<user>/n3-07b-solo.git"   # <-- sửa <user>
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git pull -q
!pip -q install open_clip_torch faiss-cpu pyarrow pyyaml

In [ ]:
# Ô 3 — chép ảnh train từ Drive sang ổ Colab rồi giải nén (đọc ảnh từ Drive trực tiếp rất chậm)
if not os.path.exists("/content/data/bdd100k/images/100k/train"):
    !mkdir -p /content/data
    !cp /content/drive/MyDrive/n3-07b/raw/100k_images_train.zip /content/
    !unzip -q -n /content/100k_images_train.zip -d /content/data && rm /content/100k_images_train.zip
!ls /content/data/bdd100k/images/100k/train | wc -l

In [ ]:
# Ô 4 — biến môi trường: ảnh ở ổ Colab, chỉ mục ghi thẳng lên Drive
os.environ["DATA_ROOT"] = "/content/data"
os.environ["INDEX_ROOT"] = "/content/drive/MyDrive/n3-07b/index"

In [ ]:
# Ô 5 — đo tốc độ trên 200 ảnh (NFR-23). Ghi số crop/s vào docs/pilot.md, rồi xoá index/vitb32_test_* trên Drive
!python -m src.search.embed --model ViT-B-32 --limit 200 --per-shard 100

In [ ]:
# Ô 6 — chạy thật 40.000 ảnh pool (280.000 crop)
!python -m src.search.embed --model ViT-B-32

In [ ]:
# Ô 7 — (W2, nếu còn quota) model thứ hai. Đo trước bằng --limit 200; > 10 giờ thì bỏ, ghi hạn chế
# !python -m src.search.embed --model ViT-B-16-SigLIP-384 --amp --limit 200 --per-shard 100
# !python -m src.search.embed --model ViT-B-16-SigLIP-384 --amp